# Notebook 8 — ML Visualisations

**Pipeline position:** runs after `5_ML_FINAL.ipynb`  
**Inputs:** `intermediary/Master.csv`, `intermediary/clusters_k5_agg.csv`  
**Outputs:** `Visualisation/charts/ml/`  

Charts produced:
- Chart 7: Feature Importance Consensus (LASSO / Ridge / Elastic Net)
- Chart 8: Standardised Coefficients
- Chart 9: Train vs Test R² comparison
- Chart 10: Predicted vs Actual ECI
- Chart 11: Random Forest Feature Importance
- Chart Z: ECI Forecast 2020–2030

# ML Visualisations

**Capstone — Moody's Ratings**

Charts for Section 3.5 (Machine Learning). Re-fits models from NB5 spec, then generates all ML figures.
Reads from `intermediary/Master.csv` and `intermediary/clusters_k5_agg.csv`.

## 0. Setup

In [1]:
import os, warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

# ── Project root ─────────────────────────────────────────────────────────────
ROOT = '/Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic'
os.chdir(ROOT)

# ── Output ───────────────────────────────────────────────────────────────────
OUT = os.path.join('Final', 'charts', 'ml')
os.makedirs(OUT, exist_ok=True)

# ── Style ────────────────────────────────────────────────────────────────────
FONT = 'Public Sans, -apple-system, BlinkMacSystemFont, sans-serif'
BG   = '#ffffff'
NAVY = '#1a2744'
GRID = '#e5e7eb'
CFG  = {'displayModeBar': False, 'responsive': True}

PAL = {'blue': '#4a6fa5', 'red': '#c23a3a', 'green': '#2e7d4a',
       'gold': '#c9a227', 'grey': '#999999', 'orange': '#d4853b'}

CLUSTER_COLORS = {
    'Petrostates':        '#E63946',
    'Oil Exporters':      '#457B9D',
    'Major Producers':    '#2A9D8F',
    'Mining Exporters':   '#E9C46A',
    'Forestry Intensive': '#8B5CF6',
}

def base_layout(**kw):
    d = dict(template='plotly_white', plot_bgcolor=BG, paper_bgcolor=BG,
             font=dict(family=FONT, size=11, color=NAVY),
             margin=dict(l=160, r=40, t=10, b=50))
    d.update(kw)
    return d

def save(fig, name, w=1100, h=600):
    path = os.path.join(OUT, name)
    fig.write_html(f"{path}.html", config=CFG)
    print(f"  Saved: {path}.html")
    try:
        fig.write_image(f"{path}.png", width=w, height=h, scale=3)
        print(f"  Saved: {path}.png")
    except Exception:
        print(f"  (PNG skipped — pip install kaleido)")

print(f'Root: {ROOT}')
print(f'Output: {OUT}')

Root: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP
Output: Final/charts/ml


In [2]:
# ── Redirect all saves to outputs/charts/ ────────────────────────────────────
import os as _os
_CHARTS = _os.path.join('Final', 'charts', 'ml')
_os.makedirs(_CHARTS, exist_ok=True)

def save(fig, name, w=1100, h=600):
    path = _os.path.join(_CHARTS, name)
    # fig.write_html(f'{path}.html')  # disabled: save disk space
    try:
        fig.write_image(f'{path}.png', width=w, height=h, scale=2)
        print(f'  Saved: {path}.png')
    except Exception as e:
        print(f'  PNG skipped: {e}')


## 1. Data & Feature Engineering (mirrors NB5)

In [3]:
master = pd.read_csv('intermediary/Master.csv')
include_list = pd.read_csv('intermediary/high_resource_countries.csv')['Country Code'].unique().tolist()
df = master[(master['Year'].between(1995, 2019)) &
            (master['Country Code'].isin(include_list))].copy()
df = df.sort_values(['Country Code', 'Year']).reset_index(drop=True)

clusters = pd.read_csv('intermediary/clusters_k5_agg.csv')
df = df.merge(clusters[['Country Code', 'ClusterLabels']].drop_duplicates('Country Code'),
              on='Country Code', how='left')

df['Total_Production_Value_Per_Capita'] = (
    df['Total_Production_Value'] / df['Population'].replace(0, np.nan))
df['L1_ECI']    = df.groupby('Country Code')['Economic Complexity Index'].shift(1)
df['ECI_delta'] = df['Economic Complexity Index'] - df['L1_ECI']
df = df.dropna(subset=['L1_ECI', 'Economic Complexity Index', 'ECI_delta'])

log_cols = [
    'Human capital index',
    'Total_Production_Value_Per_Capita',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP',
    'Government revenue',
    'Use of IMF credit (DOD, current US$)',
]
df[log_cols] = np.log1p(df[log_cols]).replace([np.inf, -np.inf], np.nan)

hci_m  = df['Human capital index'].mean()
prod_m = df['Total_Production_Value_Per_Capita'].mean()
gfcf_m = df['Gross fixed capital formation, all, Constant prices, Percent of GDP'].mean()
df['HCI_x_ProductionValue']  = (df['Human capital index'] - hci_m) * \
                                (df['Total_Production_Value_Per_Capita'] - prod_m)
df['GFCF_x_ProductionValue'] = (df['Gross fixed capital formation, all, Constant prices, Percent of GDP'] - gfcf_m) * \
                                (df['Total_Production_Value_Per_Capita'] - prod_m)

base_features = [
    'Total_Production_Value_Per_Capita', 'Human capital index',
    'Rule of law index', 'Political stability \u2014 estimate',
    'Trade (% of GDP)',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP',
    'Share of investment in GDP', 'Domestic credit to private sector (% of GDP)',
    'Landlocked', 'Urban population (% of total population)',
    'Government revenue', 'Capital depreciation rate',
    'Use of IMF credit (DOD, current US$)', 'Real interest rate (%)',
    'Inflation, consumer prices (annual %)', 'Access to electricity (% of population)',
    'Adjusted savings: gross savings (% of GNI)', 'L1_ECI',
    'Forestry rents (% of GDP)',
]
all_features = base_features + ['HCI_x_ProductionValue', 'GFCF_x_ProductionValue']
df = df.dropna(subset=all_features)

name_map = {
    'Total_Production_Value_Per_Capita': 'Production Value',
    'Human capital index': 'Human Capital',
    'Rule of law index': 'Rule of Law',
    'Political stability \u2014 estimate': 'Political Stability',
    'Trade (% of GDP)': 'Trade',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP': 'Capital Formation',
    'Share of investment in GDP': 'Investment Share',
    'Domestic credit to private sector (% of GDP)': 'Domestic Credit',
    'Landlocked': 'Landlocked',
    'Urban population (% of total population)': 'Urban Population',
    'Government revenue': 'Gov Revenue',
    'Capital depreciation rate': 'Depreciation',
    'Use of IMF credit (DOD, current US$)': 'IMF Credit',
    'Real interest rate (%)': 'Real Rate',
    'Inflation, consumer prices (annual %)': 'Inflation',
    'Access to electricity (% of population)': 'Electricity',
    'Adjusted savings: gross savings (% of GNI)': 'Gross Savings',
    'L1_ECI': 'Lagged ECI',
    'HCI_x_ProductionValue': 'HC \u00d7 Production',
    'GFCF_x_ProductionValue': 'GFCF \u00d7 Production',
}
short = [name_map.get(f, f) for f in all_features]
EXCLUDE_LABEL = 'Lagged ECI'  # trivially dominant, excluded from importance charts

print(f'Sample: {df["Country Code"].nunique()} countries, {len(df):,} obs, {len(all_features)} features')

Sample: 53 countries, 1,249 obs, 21 features


## 2. Train / Test Split & Model Fitting

In [4]:
class PanelTemporalCV:
    def __init__(self, years, n_splits=5, gap=1, min_train_years=8):
        self.years = np.asarray(years)
        uy = np.sort(np.unique(self.years))
        ec = uy[0] + min_train_years - 1
        lc = uy[-1] - gap - 1
        self.cutoffs = np.unique(np.linspace(ec, lc, n_splits).astype(int))
        self.n_splits = len(self.cutoffs)
        self.gap = gap
    def split(self, X=None, y=None, groups=None):
        for c in self.cutoffs:
            ti = np.where(self.years <= c)[0]
            vi = np.where(self.years > c + self.gap)[0]
            if len(ti) and len(vi): yield ti, vi
    def get_n_splits(self, X=None, y=None, groups=None): return self.n_splits

train_df = df[df['Year'] <= 2014].copy()
test_df  = df[df['Year'] >= 2015].copy()

y_tr_lv = train_df['Economic Complexity Index'].values
y_te_lv = test_df['Economic Complexity Index'].values
y_tr_dl = train_df['ECI_delta'].values
y_te_dl = test_df['ECI_delta'].values

scaler  = StandardScaler()
X_train = scaler.fit_transform(train_df[all_features].values)
X_test  = scaler.transform(test_df[all_features].values)

tscv = PanelTemporalCV(train_df['Year'].values, n_splits=5, gap=1, min_train_years=8)

# ── ECI level models ─────────────────────────────────────────────────────────
print('Fitting models (ECI level)...')
lasso   = LassoCV(cv=tscv, random_state=42, max_iter=10000).fit(X_train, y_tr_lv)
ridge   = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=tscv).fit(X_train, y_tr_lv)
elastic = ElasticNetCV(l1_ratio=[0.5], cv=tscv, random_state=42, max_iter=10000).fit(X_train, y_tr_lv)
rf      = RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=10,
                                 random_state=42, n_jobs=-1, oob_score=True).fit(X_train, y_tr_lv)
models_lv = {'LASSO': lasso, 'Ridge': ridge, 'Elastic Net': elastic, 'Random Forest': rf}
print(f'  LASSO \u03b1={lasso.alpha_:.4f}  Ridge \u03b1={ridge.alpha_:.4f}  '
      f'EN \u03b1={elastic.alpha_:.4f}  RF OOB={rf.oob_score_:.3f}')

# ── ΔECI models ──────────────────────────────────────────────────────────────
print('Fitting models (\u0394ECI)...')
lasso_d   = LassoCV(cv=tscv, random_state=42, max_iter=10000).fit(X_train, y_tr_dl)
ridge_d   = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=tscv).fit(X_train, y_tr_dl)
elastic_d = ElasticNetCV(l1_ratio=[0.5], cv=tscv, random_state=42, max_iter=10000).fit(X_train, y_tr_dl)
rf_d      = RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=10,
                                   random_state=42, n_jobs=-1, oob_score=True).fit(X_train, y_tr_dl)
models_dl = {'LASSO': lasso_d, 'Ridge': ridge_d, 'Elastic Net': elastic_d, 'Random Forest': rf_d}

# ── Performance tables ───────────────────────────────────────────────────────
def eval_models(models, X_tr, y_tr, X_te, y_te):
    rows = []
    for name, m in models.items():
        tr_r2 = r2_score(y_tr, m.predict(X_tr))
        te_r2 = r2_score(y_te, m.predict(X_te))
        rows.append({'Model': name, 'Train R\u00b2': round(tr_r2, 4),
                     'Test R\u00b2': round(te_r2, 4),
                     'Overfit Gap': round(tr_r2 - te_r2, 4)})
    return pd.DataFrame(rows).sort_values('Test R\u00b2', ascending=False).reset_index(drop=True)

perf_lv = eval_models(models_lv, X_train, y_tr_lv, X_test, y_te_lv)
perf_dl = eval_models(models_dl, X_train, y_tr_dl, X_test, y_te_dl)

print('\n-- ECI Level --')
print(perf_lv.to_string(index=False))
print('\n-- \u0394ECI --')
print(perf_dl.to_string(index=False))

# ── Normalised importance ────────────────────────────────────────────────────
def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo + 1e-12)

imp = pd.DataFrame({'Feature': all_features, 'Short': short})
imp['LASSO']       = minmax(np.abs(lasso.coef_))
imp['Ridge']       = minmax(np.abs(ridge.coef_))
imp['Elastic Net'] = minmax(np.abs(elastic.coef_))
imp['RF']          = minmax(rf.feature_importances_)
imp['avg']         = imp[['LASSO', 'Ridge', 'Elastic Net', 'RF']].mean(axis=1)
imp_show = imp[imp['Short'] != EXCLUDE_LABEL].sort_values('avg', ascending=False).reset_index(drop=True)

print(f'\nModels fitted. Train: {len(train_df):,} obs | Test: {len(test_df):,} obs')

Fitting models (ECI level)...


  LASSO α=0.0294  Ridge α=0.0010  EN α=0.0548  RF OOB=0.768
Fitting models (ΔECI)...



-- ECI Level --
        Model  Train R²  Test R²  Overfit Gap
Random Forest    0.8185   0.8703      -0.0518
        LASSO    0.7698   0.8576      -0.0878
  Elastic Net    0.7703   0.8545      -0.0842
        Ridge    0.7844   0.8446      -0.0602

-- ΔECI --
        Model  Train R²  Test R²  Overfit Gap
Random Forest    0.2363   0.1506       0.0857
  Elastic Net    0.0989   0.1035      -0.0046
        LASSO    0.0965   0.1020      -0.0056
        Ridge    0.1396   0.0945       0.0451

Models fitted. Train: 984 obs | Test: 265 obs


## Fig 7 — Feature Importance Consensus (LASSO / Ridge / Elastic Net)

In [5]:
d = imp_show.head(15).iloc[::-1].reset_index(drop=True)
lin_models = ['LASSO', 'Ridge', 'Elastic Net']
symbols = {'LASSO': 'circle', 'Ridge': 'square', 'Elastic Net': 'triangle-up'}
colors  = {'LASSO': PAL['red'], 'Ridge': PAL['blue'], 'Elastic Net': PAL['green']}

fig7 = go.Figure()
for _, row in d.iterrows():
    vals = [row[m] for m in lin_models if not np.isnan(row[m])]
    if len(vals) >= 2:
        fig7.add_shape(type='line',
                       x0=min(vals), x1=max(vals), y0=row['Short'], y1=row['Short'],
                       line=dict(color='#b0c0d4', width=2))

for m in lin_models:
    fig7.add_trace(go.Scatter(
        x=d[m], y=d['Short'], mode='markers', name=m,
        marker=dict(symbol=symbols[m], color=colors[m], size=11,
                    line=dict(width=1.2, color='white')),
        hovertemplate=f'%{{y}}: %{{x:.3f}}<extra>{m}</extra>',
    ))

fig7.update_layout(**base_layout(
    height=560, margin=dict(l=180, r=60, t=40, b=60),
    xaxis=dict(title='Normalised Feature Importance (min-max, 0\u20131)',
               gridcolor=GRID, gridwidth=0.5, range=[-0.02, 0.27]),
    yaxis=dict(tickfont=dict(size=11)),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5,
                font=dict(size=11)),
))
save(fig7, '07_ml_feature_importance_consensus', w=1200, h=560)
fig7.show(config=CFG)

  Saved: Final/charts/ml/07_ml_feature_importance_consensus.png


## Fig 8 — Standardised Coefficients (LASSO / Ridge / Elastic Net)

In [6]:
coef_df = pd.DataFrame({'Feature': all_features, 'Short': short})
coef_df['LASSO']       = lasso.coef_
coef_df['Ridge']       = ridge.coef_
coef_df['Elastic Net'] = elastic.coef_
coef_df = coef_df[coef_df['Short'] != EXCLUDE_LABEL].copy()
coef_df['abs_avg'] = coef_df[['LASSO', 'Ridge', 'Elastic Net']].abs().mean(axis=1)
coef_df = coef_df.sort_values('abs_avg', ascending=True).reset_index(drop=True)

figC = go.Figure()
figC.add_vline(x=0, line=dict(color='#c9cfd6', width=1.5))
for m, col in [('LASSO', PAL['red']), ('Ridge', PAL['blue']), ('Elastic Net', PAL['green'])]:
    figC.add_trace(go.Bar(
        y=coef_df['Short'], x=coef_df[m], name=m, orientation='h',
        marker=dict(color=col, opacity=0.88, line=dict(color='white', width=0.5)),
        hovertemplate=f'%{{y}}: %{{x:+.3f}}<extra>{m}</extra>',
    ))

figC.update_layout(**base_layout(
    height=560, barmode='group', margin=dict(l=170, r=60, t=40, b=60),
    xaxis=dict(title='Coefficient (standardised inputs)', gridcolor=GRID,
               zeroline=False),
    yaxis=dict(tickfont=dict(size=11)),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
))
save(figC, '08_ml_standardised_coefficients', w=1100, h=560)
figC.show(config=CFG)

  Saved: Final/charts/ml/08_ml_standardised_coefficients.png


## Fig 11 — Random Forest Feature Importance

In [7]:
rf_show = imp_show.sort_values('RF', ascending=True).copy()

fig11 = go.Figure(go.Bar(
    y=rf_show['Short'], x=rf_show['RF'], orientation='h',
    marker=dict(color='#c97030', opacity=0.9, line=dict(color='white', width=0.5)),
    hovertemplate='%{y}: %{x:.3f}<extra>Random Forest</extra>',
))
fig11.update_layout(**base_layout(
    height=500, margin=dict(l=170, r=60, t=40, b=60),
    xaxis=dict(title='Feature Importance (MDI)', gridcolor=GRID),
    yaxis=dict(tickfont=dict(size=11)),
    showlegend=False,
))
save(fig11, '11_ml_random_forest_importance', w=1100, h=500)
fig11.show(config=CFG)

  Saved: Final/charts/ml/11_ml_random_forest_importance.png


## Fig 9 — Train vs Test R\u00b2 (ECI level and \u0394ECI)

In [8]:
fig9 = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                     subplot_titles=['ECI Level', '\u0394ECI'])

for col_idx, perf in enumerate([perf_lv, perf_dl], 1):
    show = (col_idx == 1)
    fig9.add_trace(go.Bar(
        x=perf['Model'], y=perf['Train R\u00b2'], name='Train R\u00b2',
        marker_color=PAL['blue'], opacity=0.85, legendgroup='train',
        text=[f'{v:.3f}' for v in perf['Train R\u00b2']], textposition='outside',
        textfont=dict(size=10, color=PAL['blue']),
        showlegend=show,
    ), row=1, col=col_idx)
    fig9.add_trace(go.Bar(
        x=perf['Model'], y=perf['Test R\u00b2'], name='Test R\u00b2',
        marker_color=PAL['red'], opacity=0.85, legendgroup='test',
        text=[f'{v:.3f}' for v in perf['Test R\u00b2']], textposition='outside',
        textfont=dict(size=10, color=PAL['red']),
        showlegend=show,
    ), row=1, col=col_idx)
    fig9.update_xaxes(tickangle=-25, tickfont=dict(size=10), row=1, col=col_idx)
    fig9.update_yaxes(title_text='R\u00b2' if col_idx == 1 else '',
                      gridcolor=GRID, gridwidth=0.5, row=1, col=col_idx)

fig9.update_layout(
    template='plotly_white', plot_bgcolor=BG, paper_bgcolor=BG,
    font=dict(family=FONT, size=11, color=NAVY),
    barmode='group', height=480, margin=dict(l=60, r=40, t=50, b=80),
    legend=dict(orientation='h', yanchor='bottom', y=1.06, xanchor='center', x=0.5,
                font=dict(size=12)),
)
save(fig9, '09_ml_train_vs_test_r2', w=1200, h=480)
fig9.show(config=CFG)

  Saved: Final/charts/ml/09_ml_train_vs_test_r2.png


## Fig 10 — Predicted vs Actual ECI (Elastic Net, test set)

In [9]:

pred_lv = elastic.predict(X_test)
pred_dl = elastic_d.predict(X_test)

HIGHLIGHT_CC = ['CHL', 'AZE', 'COG']
HL_COL       = PAL['orange']
DOT_COL      = PAL['blue']

def make_fig10(label_highlights):
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                        subplot_titles=['ECI Level', '\u0394ECI'])

    codes = test_df['Country Code'].values
    names = test_df['Country Name'].values

    for col_idx, (actual, pred, lbl) in enumerate([
        (y_te_lv, pred_lv, 'ECI'),
        (y_te_dl, pred_dl, '\u0394ECI'),
    ], 1):
        lims = [min(actual.min(), pred.min()) - 0.15,
                max(actual.max(), pred.max()) + 0.15]

        # 45-degree line
        fig.add_trace(go.Scatter(
            x=lims, y=lims, mode='lines',
            line=dict(color='#999', width=1.5, dash='dash'),
            showlegend=False, hoverinfo='skip',
        ), row=1, col=col_idx)

        # Quadrant shading
        mid = 0.0
        for x0, x1, y0, y1, fc in [
            (lims[0], mid, lims[0], mid, 'rgba(46,125,74,0.06)'),
            (mid, lims[1], mid, lims[1], 'rgba(46,125,74,0.06)'),
            (lims[0], mid, mid, lims[1], 'rgba(194,58,58,0.06)'),
            (mid, lims[1], lims[0], mid, 'rgba(194,58,58,0.06)'),
        ]:
            fig.add_shape(type='rect', x0=x0, x1=x1, y0=y0, y1=y1,
                          fillcolor=fc, line=dict(width=0), layer='below',
                          row=1, col=col_idx)

        # Split: highlight vs regular
        hl_mask  = np.array([c in HIGHLIGHT_CC for c in codes])
        reg_mask = ~hl_mask

        # Regular points — single neutral color
        fig.add_trace(go.Scatter(
            x=actual[reg_mask], y=pred[reg_mask], mode='markers',
            marker=dict(size=5, color=DOT_COL, opacity=0.45,
                        line=dict(color='white', width=0.4)),
            name='Countries', legendgroup='reg', showlegend=(col_idx == 1),
            customdata=np.stack([codes[reg_mask], names[reg_mask]], axis=1),
            hovertemplate='<b>%{customdata[1]}</b><br>Actual: %{x:.3f}<br>Pred: %{y:.3f}<extra></extra>',
        ), row=1, col=col_idx)

        # Highlighted countries
        if label_highlights and hl_mask.sum() > 0:
            fig.add_trace(go.Scatter(
                x=actual[hl_mask], y=pred[hl_mask],
                mode='markers+text',
                marker=dict(size=9, color=HL_COL, opacity=0.9,
                            line=dict(color='white', width=1)),
                text=codes[hl_mask], textposition='top center',
                textfont=dict(size=8, color=HL_COL),
                name='CHL / AZE / COG', legendgroup='hl',
                showlegend=(col_idx == 1),
                hovertemplate='<b>%{text}</b><br>Actual: %{x:.3f}<br>Pred: %{y:.3f}<extra></extra>',
            ), row=1, col=col_idx)

        r2_val = r2_score(actual, pred)
        fig.add_annotation(
            x=0.04 + (col_idx - 1) * 0.52, y=0.96,
            xref='paper', yref='paper',
            text=f'<b>R\u00b2 = {r2_val:.3f}</b>', showarrow=False,
            font=dict(size=12, color=PAL['blue']),
            bgcolor='rgba(255,255,255,0.8)',
        )
        fig.update_xaxes(title_text=f'Actual {lbl}', range=lims,
                         gridcolor=GRID, gridwidth=0.5, row=1, col=col_idx)
        fig.update_yaxes(title_text=f'Predicted {lbl}', range=lims,
                         gridcolor=GRID, gridwidth=0.5, row=1, col=col_idx)

    fig.update_layout(
        template='plotly_white', plot_bgcolor=BG, paper_bgcolor=BG,
        font=dict(family=FONT, size=11, color=NAVY),
        height=560, margin=dict(l=70, r=40, t=50, b=70),
        legend=dict(orientation='h', yanchor='bottom', y=1.04,
                    xanchor='center', x=0.5, font=dict(size=10)),
    )
    return fig

fig10a = make_fig10(label_highlights=False)
save(fig10a, '10a_ml_predicted_vs_actual', w=1200, h=560)

fig10b = make_fig10(label_highlights=True)
save(fig10b, '10b_ml_predicted_vs_actual_labeled', w=1200, h=560)


  Saved: Final/charts/ml/10a_ml_predicted_vs_actual.png


  Saved: Final/charts/ml/10b_ml_predicted_vs_actual_labeled.png


## Fig Z — ECI Forecast 2020\u20132030 (Top Improvers & Decliners)

In [10]:

# ── Retrain on full sample ────────────────────────────────────────────────────
print('Retraining on full 1995-2019 sample for forecasting...')
X_full_raw = df[all_features].values
y_full     = df['Economic Complexity Index'].values
scaler_full = StandardScaler()
X_full      = scaler_full.fit_transform(X_full_raw)
full_years  = df['Year'].values
tscv_full   = PanelTemporalCV(full_years, n_splits=5, gap=1, min_train_years=8)

fc_lasso   = LassoCV(cv=tscv_full, random_state=42, max_iter=10000).fit(X_full, y_full)
fc_ridge   = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=tscv_full).fit(X_full, y_full)
fc_elastic = ElasticNetCV(l1_ratio=[0.5], cv=tscv_full, random_state=42, max_iter=10000).fit(X_full, y_full)
fc_rf      = RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=10,
                                    random_state=42, n_jobs=-1).fit(X_full, y_full)
fc_models  = {'LASSO': fc_lasso, 'Ridge': fc_ridge, 'Elastic Net': fc_elastic, 'Random Forest': fc_rf}
print('  Done.')

FORECAST_YEARS = list(range(2020, 2031))
trend_features = [f for f in base_features if f != 'L1_ECI']

def extrapolate_country(cdf, years):
    last5 = cdf.tail(5)
    rows = []
    for yr in years:
        row = {'Year': yr, 'Country Code': cdf['Country Code'].iloc[0],
               'Country Name': cdf['Country Name'].iloc[0]}
        for feat in trend_features:
            vals = last5[feat].dropna().values
            if len(vals) >= 2:
                slope, intercept = np.polyfit(np.arange(len(vals)), vals, 1)
                steps = yr - int(last5['Year'].iloc[-1])
                row[feat] = float(intercept + slope * (len(vals) - 1 + steps))
            else:
                row[feat] = float(vals[-1]) if len(vals) else 0.0
        rows.append(row)
    return rows

future_rows = []
for cc, cdf in df.groupby('Country Code'):
    future_rows.extend(extrapolate_country(cdf, FORECAST_YEARS))
future_df = pd.DataFrame(future_rows)

# ── Iterative forecast ────────────────────────────────────────────────────────
records = []
for cc, cdf in df.groupby('Country Code'):
    last_eci = float(cdf.sort_values('Year')['Economic Complexity Index'].iloc[-1])
    fsub = future_df[future_df['Country Code'] == cc].sort_values('Year')
    running = {n: last_eci for n in fc_models}

    for _, frow in fsub.iterrows():
        preds = {}
        for name, model in fc_models.items():
            row_data = frow.copy()
            row_data['L1_ECI'] = running[name]
            row_data['HCI_x_ProductionValue'] = (
                (row_data['Human capital index'] - hci_m) *
                (row_data['Total_Production_Value_Per_Capita'] - prod_m))
            row_data['GFCF_x_ProductionValue'] = (
                (row_data['Gross fixed capital formation, all, Constant prices, Percent of GDP'] - gfcf_m) *
                (row_data['Total_Production_Value_Per_Capita'] - prod_m))
            x_vec = np.array([row_data[f] for f in all_features]).reshape(1, -1)
            pred  = model.predict(scaler_full.transform(x_vec))[0]
            preds[name] = pred
            running[name] = pred

        rec = {'Country Code': cc, 'Country Name': frow['Country Name'],
               'Year': frow['Year'], 'Last_Known_ECI': last_eci,
               'Ensemble': np.mean(list(preds.values()))}
        rec.update(preds)
        records.append(rec)

forecast_df = pd.DataFrame(records)
model_cols = ['LASSO', 'Ridge', 'Elastic Net', 'Random Forest']
forecast_df['ECI_std'] = forecast_df[model_cols].std(axis=1)

ranking = forecast_df.groupby('Country Code').agg(
    Country=('Country Name', 'first'),
    ECI_2019=('Last_Known_ECI', 'first'),
    ECI_2030=('Ensemble', 'last'),
).reset_index()
ranking['Total_Change'] = ranking['ECI_2030'] - ranking['ECI_2019']
ranking = ranking.sort_values('Total_Change', ascending=False).reset_index(drop=True)

top3    = ranking.head(3)['Country Code'].tolist()
bottom3 = ranking.tail(3)['Country Code'].tolist()
print(f'Top 3 improvers: {top3}')
print(f'Bottom 3 decliners: {bottom3}')

# ── Plot ──────────────────────────────────────────────────────────────────────
hist    = df[['Country Code', 'Country Name', 'Year', 'Economic Complexity Index']].dropna()
all_cc  = df['Country Code'].unique().tolist()

TOP_COL  = '#2e7d4a'
BOT_COL  = '#c23a3a'
GREY_LINE = '#9aafc4'

def hex_rgba(hex_col, alpha):
    h = hex_col.lstrip('#')
    r, g, b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    return f'rgba({r},{g},{b},{alpha})'

figZ = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                     shared_yaxes=True)

for panel, highlight, h_col in [(1, top3, TOP_COL), (2, bottom3, BOT_COL)]:

    # Background grey lines (historical only)
    for cc in all_cc:
        if cc in highlight: continue
        h = hist[hist['Country Code'] == cc].sort_values('Year')
        if len(h) == 0: continue
        figZ.add_trace(go.Scatter(
            x=h['Year'], y=h['Economic Complexity Index'], mode='lines',
            line=dict(color=GREY_LINE, width=0.6), opacity=0.25,
            showlegend=False, hoverinfo='skip',
        ), row=1, col=panel)

    # Highlighted countries
    for cc in highlight:
        h = hist[hist['Country Code'] == cc].sort_values('Year')
        f = forecast_df[forecast_df['Country Code'] == cc].sort_values('Year')
        if len(h) == 0: continue

        last_y = h['Economic Complexity Index'].iloc[-1]
        last_x = h['Year'].iloc[-1]

        # Historical solid
        figZ.add_trace(go.Scatter(
            x=h['Year'], y=h['Economic Complexity Index'], mode='lines',
            line=dict(color=h_col, width=2.0),
            showlegend=False, hoverinfo='skip',
        ), row=1, col=panel)

        if len(f) == 0: continue
        f = f.sort_values('Year')
        bx = [last_x] + f['Year'].tolist()
        by = [last_y] + f['Ensemble'].tolist()
        bstd = [0]    + f['ECI_std'].tolist()

        # CI band
        upper = [y + s for y, s in zip(by, bstd)]
        lower = [y - s for y, s in zip(by, bstd)]
        figZ.add_trace(go.Scatter(
            x=bx + bx[::-1], y=upper + lower[::-1],
            fill='toself', fillcolor=hex_rgba(h_col, 0.12),
            line=dict(width=0), showlegend=False, hoverinfo='skip',
        ), row=1, col=panel)

        # Forecast dashed
        figZ.add_trace(go.Scatter(
            x=bx, y=by, mode='lines',
            line=dict(color=h_col, width=1.8, dash='dash'),
            showlegend=False, hoverinfo='skip',
        ), row=1, col=panel)

        # Country label at right end
        xref = 'x' if panel == 1 else 'x2'
        yref = 'y' if panel == 1 else 'y2'
        figZ.add_annotation(
            x=2030, y=by[-1], xref=xref, yref=yref,
            text=f'<b>{cc}</b>', showarrow=True,
            ax=18, ay=0, arrowwidth=1, arrowcolor=h_col,
            font=dict(size=9, color=h_col), xanchor='left',
        )

    figZ.add_vline(x=2019.5, line=dict(color='#aaa', width=1.2, dash='dot'),
                   row=1, col=panel)
    figZ.update_xaxes(title_text='Year', gridcolor=GRID, gridwidth=0.5,
                       dtick=5, range=[1994, 2033], row=1, col=panel)

figZ.update_yaxes(title_text='Economic Complexity Index',
                   gridcolor=GRID, gridwidth=0.5, row=1, col=1)
figZ.update_yaxes(gridcolor=GRID, gridwidth=0.5, row=1, col=2)

# Legend
top_names = ' \u00b7 '.join(top3)
bot_names = ' \u00b7 '.join(bottom3)
for lbl, col, dash, rk in [
    (f'Top 3 improvers \u2014 {top_names}', TOP_COL,  'solid', 1),
    (f'Bottom 3 decliners \u2014 {bot_names}', BOT_COL, 'solid', 2),
    ('Other countries',                       GREY_LINE,'solid', 3),
    ('Historical',                            '#555',   'solid', 4),
    ('Forecast',                              '#555',   'dash',  5),
]:
    figZ.add_trace(go.Scatter(
        x=[None], y=[None], mode='lines',
        line=dict(color=col, width=2.2, dash=dash),
        name=lbl, legendrank=rk,
    ))

figZ.update_layout(
    template='plotly_white', plot_bgcolor=BG, paper_bgcolor=BG,
    font=dict(family=FONT, size=11, color=NAVY),
    height=600, margin=dict(l=70, r=50, t=30, b=90),
    legend=dict(
        orientation='h', font=dict(size=10),
        bgcolor='rgba(255,255,255,0.92)', bordercolor=GRID, borderwidth=1,
        x=0.0, y=-0.14, xanchor='left', yanchor='top',
    ),
)
save(figZ, '12_ml_eci_forecast_2020_2030', w=1300, h=600)
figZ.show(config=CFG)


Retraining on full 1995-2019 sample for forecasting...


  Done.


Top 3 improvers: ['GNQ', 'CHL', 'MNG']
Bottom 3 decliners: ['KAZ', 'MYS', 'SAU']


  Saved: Final/charts/ml/12_ml_eci_forecast_2020_2030.png


## Summary

In [11]:
print('=' * 60)
print('ML Visualisations complete.')
print('=' * 60)
print(f'Sample: {df["Country Code"].nunique()} countries, {len(df):,} obs')
print(f'Train: {len(train_df):,} (1996-2014) | Test: {len(test_df):,} (2015-2019)')
print(f'Features: {len(all_features)} ({len(base_features)} base + 2 interactions)')
print(f'\nCharts saved to: {OUT}/')
print('  07_ml_feature_importance_consensus')
print('  08_ml_standardised_coefficients')
print('  09_ml_train_vs_test_r2')
print('  10_ml_predicted_vs_actual')
print('  11_ml_random_forest_importance')
print('  12_ml_eci_forecast_2020_2030')

ML Visualisations complete.
Sample: 53 countries, 1,249 obs
Train: 984 (1996-2014) | Test: 265 (2015-2019)
Features: 21 (19 base + 2 interactions)

Charts saved to: Final/charts/ml/
  07_ml_feature_importance_consensus
  08_ml_standardised_coefficients
  09_ml_train_vs_test_r2
  10_ml_predicted_vs_actual
  11_ml_random_forest_importance
  12_ml_eci_forecast_2020_2030
